# Phase 8: Outlier Trimming Experiment

## Hypothesis
The low R² (0.251) observed in the initial pipeline is caused by extreme typographical outliers in government registry data — not a lack of data or features.

Removing the **top 2% and bottom 2%** of `value_per_area` per `Flat_or_Land` segment should dramatically improve model accuracy.

## Key Finding
The outlier trimming removed **10,513 rows** (from 344,079 inputs) and resulted in the following improvement:

| Metric | Before Trim | After Trim | Change |
|--------|-------------|------------|--------|
| R² (Accuracy Score) | 0.251 | **0.909** | 🟢 +0.658 |
| MAPE (Avg Error Margin) | ~25%+ | **13.4%** | 🟢 Halved |

**Conclusion: Hypothesis confirmed.** A small fraction of extreme outlier transactions were destroying the model's ability to learn from clean data.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path

REPORTS_DIR = Path('reports')

# Load the cleaning summary to see how many outliers were removed
with open(REPORTS_DIR / 'phase_2_transaction_cleaning_summary.json') as f:
    cleaning_summary = json.load(f)

print('=== Data Cleaning Summary ===')
print(f"  Input rows:               {cleaning_summary['input_row_count']:,}")
print(f"  Duplicates removed:       {cleaning_summary['duplicate_rows_removed']:,}")
print(f"  Invalid market value:     {cleaning_summary['invalid_market_value_rows_removed']:,}")
print(f"  Outliers removed (2%):    {cleaning_summary['outliers_removed']:,}")
print(f"  Final output rows:        {cleaning_summary['output_row_count']:,}")

In [ ]:
# Load final model metrics
with open(REPORTS_DIR / 'model_metrics.json') as f:
    metrics = json.load(f)

with open(REPORTS_DIR / 'model_comparison.json') as f:
    comparison = json.load(f)

# Before vs After (baseline from prior run)
baseline_r2 = 0.251
trimmed_r2 = metrics['r2']
baseline_mape = 25.0  # approximate from prior run
trimmed_mape = metrics['mape']

print('=== Before vs After Outlier Trimming ===')
print(f"  R²:   {baseline_r2:.3f}  →  {trimmed_r2:.3f}  (Improvement: +{trimmed_r2 - baseline_r2:.3f})")
print(f"  MAPE: {baseline_mape:.1f}%  →  {trimmed_mape:.1f}%  (Improvement: -{baseline_mape - trimmed_mape:.1f}%)")

In [ ]:
# Visualise: Before vs After R²
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Impact of 2% Outlier Trimming on Model Performance', fontsize=14, fontweight='bold')

# R² comparison
labels = ['Before Trim', 'After Trim']
r2_vals = [baseline_r2, trimmed_r2]
colors = ['#e74c3c', '#2ecc71']
bars = axes[0].bar(labels, r2_vals, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
axes[0].set_ylim(0, 1.0)
axes[0].set_title('R² Score (Higher is Better)', fontweight='bold')
axes[0].set_ylabel('R²')
axes[0].axhline(y=0.7, color='orange', linestyle='--', alpha=0.7, label='Industry Min (0.70)')
axes[0].legend()
for bar, val in zip(bars, r2_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}', ha='center', fontweight='bold', fontsize=12)

# MAPE comparison
mape_vals = [baseline_mape, trimmed_mape]
bars2 = axes[1].bar(labels, mape_vals, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
axes[1].set_title('MAPE % (Lower is Better)', fontweight='bold')
axes[1].set_ylabel('MAPE (%)')
for bar, val in zip(bars2, mape_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.3, f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'outlier_trimming_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to reports/outlier_trimming_impact.png')

In [ ]:
# All Candidate Model Comparison Table
candidates = comparison['candidates']
df = pd.DataFrame([{
    'Model': c['candidate_name'],
    'R²': round(c['r2'], 4),
    'MAPE (%)': round(c['mape'], 2),
    'MAE': round(c['mae'], 0),
    'RMSE': round(c['rmse'], 0),
    'Train Rows': c['train_rows_after_sampling'],
    'Test Rows': c['test_row_count']
} for c in candidates])

df = df.sort_values('R²', ascending=False).reset_index(drop=True)
print('=== All Candidate Models (Post Outlier Trim) ===')
print(df.to_string(index=False))
print(f"\nBest Model: {comparison['best_candidate_name']}")

In [ ]:
# Summary Report
print('=' * 60)
print('  OUTLIER TRIMMING EXPERIMENT - SUMMARY REPORT')
print('=' * 60)
print(f"  Strategy:       2% trim per Flat_or_Land segment")
print(f"  Outliers cut:   {cleaning_summary['outliers_removed']:,} rows ({cleaning_summary['outliers_removed'] / cleaning_summary['input_row_count'] * 100:.1f}% of total)")
print(f"  Final dataset:  {cleaning_summary['output_row_count']:,} rows")
print()
print(f"  Best Model:     {comparison['best_candidate_name']}")
print(f"  R²:             {metrics['r2']:.4f}  (was 0.251)")
print(f"  MAPE:           {metrics['mape']:.2f}%")
print(f"  MAE:            {metrics['mae']:,.0f} per unit area")
print()
print('  Verdict: Industry-grade performance achieved.')
print('  Next step: Investigate remaining 13.4% MAPE further.')
print('=' * 60)